# Data Preparation
In this section, we walk through the steps needed to organize your data into the format required by our functions. An overview of the workflow is illustrated in the schematic below.

```{image} pic/data_prep/fig1.png
:alt: Overall data preparation workflow
:width: 600px
```

``fMRI-HA`` is a Python-based toolbox, and most of its computational routines operate on data stored in **NumPy (`.npy`) format**. Therefore, an essential prerequisite for using ``fMRI-HA``  is to preprocess your data by loading it from its original format and saving it as `.npy` arrays that conform to the required input structure. And the saved matrices are usually arranged as `time points x vertices/voxels`.

In addition to data format, ``fMRI-HA`` follows a **BIDS-inspired naming and directory convention** to organize files in a clear and consistent manner. File names are constructed using key–value pairs (e.g., `key1-value1_key2-value2_...`), which makes metadata explicit and facilitates automated processing.

Typical output hierarchy:

```text
work_dir/
  sub-<subject>/
    resample/
      response/
        hemi-L/
          sub-<subject>_ses-01_run-01_space-surf_hemi-L_roi-cortical_res-onavg32_desc-bold-demean_set-demo.npy
        hemi-R/
        volume-subcortical-L/
        volume-subcortical-R/
        volume-subcortical-BRAIN_STEM/
        volume-<roi_name>/
  logs/
    prep_log.json
    progress_<log_num>.log
  masks/
```

## Scripts Workflow

Begin by importing the necessary packages. Replace all example paths with paths on your machine before running the code.

In [ ]:
import joblib
import numpy as np
import neuroboros as nb
from pathlib import Path
from fmriha.preprocessing import prep

### Input Validation Parameters

CIFTI and NIfTI files are validated with `prep.input_validation_nii`. GIFTI files are validated with `prep.input_validation_gii`.

| Parameter | Meaning |
|---|---|
| `file_list` | Input files. For CIFTI/NIfTI, use a list of file paths. For GIFTI, use a list of dictionaries such as `{'l': left_file, 'r': right_file}`. |
| `ses` | Session label written into output filenames as `ses-<label>`. It can represent a session, task, run group, or another dataset label. |
| `bids_parse` | If `True`, parses BIDS-like filename entities such as `sub`, `run`, `task`, and `ses`. |
| `subject_key` | Used when `bids_parse=False`. It identifies the prefix used to find subject IDs in non-BIDS filenames. |
| `check_dim` | If `True`, checks input dimensions for consistency. With `group_by`, dimensions are checked within each group. |
| `group_by` | Filename entity used to group files before dimension checking, commonly `run`. Use `None` when all files should be checked together. |
| `role` | Dataset role written into output filenames as `set-<role>`, commonly `train`, `test`, or `align`. |
| `njobs` | Number of parallel workers used during validation. Larger values can speed up I/O but require more memory. |
| `verbose` | If `True`, prints validation progress and detected shapes. |

`prep.write_json_input_validate` records validation results in `prep_log.json`. Use `overwrite=True` only when you want to clear previous records in that JSON file.

### CIFTI Surface Data

CIFTI cortical data are extracted from `.dtseries.nii` or `.dscalar.nii` files and prepared as separate left and right surface matrices. This workflow uses two functions:

1. `prep.input_validation_nii` validates the CIFTI files.
2. `prep.prep_surf` extracts cortical data, optionally resamples it to a target surface, applies a cortical or ROI mask, normalizes it, and saves `.npy` files.

For CIFTI surface preparation, `prep.prep_surf` internally loads cortical data from the CIFTI file, patches the medial wall, applies the mapper if requested, and then applies the target mask.

In [ ]:
cifti_dir = Path("/path/to/raw_data/cifti")
work_dir = Path("/path/to/workdir_cifti")
json_path = work_dir / "logs" / "prep_log.json"
sub_list = ["sub-rid000005", "sub-rid000014", "sub-rid000029"]

If your data are stored in BIDS format, set ``bids_parse=True`` and ``subject_key=None``

In [ ]:
data_list_bids = [
    cifti_dir / f"{sub}_ses-raiders_run-01_TS.dtseries.nii"
    for sub in sub_list
]

data_info_bids = prep.input_validation_nii(
    file_list=data_list_bids,
    ses="raiders",
    bids_parse=True,
    subject_key=None,
    check_dim=True,
    group_by="run",
    role="demo",
    njobs=2,
    verbose=True,
)

prep.write_json_input_validate(data_info_bids, json_path, overwrite=True)

If your data are not stored in BIDS format, set ``bids_parse=False`` and specify ``subject_key`` as the subject identifier in your filenames (e.g., ``xSfrNGRd`` in the example below).

In [ ]:
data_list = [
    cifti_dir / f"xSfrNGRd_{sub.split('-')[1]}_raiders_run-02_TS.dtseries.nii"
    for sub in sub_list
]

data_info = prep.input_validation_nii(
    file_list=data_list,
    ses="raiders",
    bids_parse=False,
    subject_key="xSfrNGRd",
    check_dim=True,
    group_by="run",
    role="demo",
    njobs=2,
    verbose=True,
)

prep.write_json_input_validate(data_info, json_path, overwrite=False)

Once the subject information has been constructed, the cortical component is extracted from the CIFTI files and processed accordingly.

Parameters of `prep.prep_surf` for CIFTI cortical data:

| Parameter | Meaning |
|---|---|
| `work_dir` | Root output directory. Prepared files are saved under subject folders inside this directory. |
| `input_info` | Validation dictionary returned by `prep.input_validation_nii`. |
| `do_resample` | If `True`, applies a surface-to-surface mapper before masking. Use `False` when source and target spaces already match. |
| `res_param` | Dictionary describing surface mapping. Required fields used here are `source`, `source_mask`, `target`, `target_mask`, `data`, `desc`, and `res`. |
| `res_param['source']` | Source surface space of the input data, such as `fslr-ico57`. |
| `res_param['source_mask']` | Source-space mask passed to the mapper. Use `None` when the mapper expects the full source surface. |
| `res_param['target']` | Target surface space, such as `onavg-ico32`. |
| `res_param['target_mask']` | Target-space mask passed to the mapper. Use `None` when masking is handled later by `medial_param`. |
| `res_param['data']` | Mapper object, usually a dictionary like `{'l': left_mapper, 'r': right_mapper}`. If `None`, the function attempts to load a mapper internally. |
| `res_param['desc']` | Human-readable description written to logs. |
| `res_param['res']` | Short resolution label written into output filenames as `res-<label>`. |
| `do_medial_rm` | If `True`, applies the mask in `medial_param`. This can remove the medial wall or extract a surface ROI. |
| `medial_param` | Dictionary describing the target-surface mask. Required fields used here are `space`, `data`, `roi_name`, and `desc`. |
| `medial_param['space']` | Surface space used when loading a mask through neuroboros. |
| `medial_param['data']` | Mask data. Use a dictionary such as `{'l': bool_array, 'r': bool_array}` where `True` means keep the vertex. |
| `medial_param['roi_name']` | ROI label written into output filenames as `roi-<label>`. |
| `medial_param['desc']` | Human-readable mask description written to logs. |
| `normalize` | Normalization method. Use `zscore`, `demean`, or `None` for raw values. |
| `dtype` | Numeric type for saved arrays, usually `np.float32` to reduce storage and memory. |
| `njobs` | Number of parallel workers used for file-wise preparation. |
| `verbose` | Joblib verbosity level. Higher values print more progress information. |
| `json_path` | JSON log path used to record processing settings. |
| `overwrite` | If `True`, overwrites matching JSON records. Use `False` to append records. |
| `log_num` | Identifier appended to the progress log filename. |

To use the built-in mapping functions, set ``source`` and ``target`` to the corresponding string identifiers, and set ``source_mask``, ``target_mask`` and ``data`` to ``None``. In the example below (``fslr-ico57`` → ``onavg-ico32``), ``source`` is the original space and ``target`` is the target space. We recommend mapping to ``onavg-ico32`` for subsequent HA analysis.

In [ ]:
res_param = {
    "source": "fslr-ico57",
    "source_mask": None,
    "target": "onavg-ico32",
    "target_mask": None,
    "data": None,
    "desc": "fslr-ico57 whole to onavg-ico32 whole",
    "res": "onavg32",
}

medial_param = {
    "space": "onavg-ico32",
    "data": None,
    "roi_name": "cortical",
    "desc": "onavg-ico32 cortical",
}

prep.prep_surf(
        work_dir,
        input_info=data_info,
        do_resample=True,
        res_param=res_param,
        do_medial_rm=True,
        medial_param=medial_param,
        normalize="zscore",
        dtype=np.float32,
        njobs=3,
        verbose=5,
        json_path=json_path,
        overwrite=False,
        log_num='cifti_demo',
)


If you prefer to use your own mapper and masks, you can configure them as follows:

In [ ]:
mapper_data = joblib.load("/path/to/mapper.pkl")
cortical_mask_data = joblib.load("/path/to/cortical_mask.pkl")

res_param = {
    "source": "fslr-ico57",
    "source_mask": None,
    "target": "onavg-ico32",
    "target_mask": None,
    "data": mapper_data,
    "desc": "fslr-ico57 whole to onavg-ico32 whole",
    "res": "onavg32",
}

medial_param = {
    "space": "onavg-ico32",
    "data": cortical_mask_data,
    "roi_name": "cortical",
    "desc": "onavg-ico32 cortical",
}

prep.prep_surf(
        work_dir,
        input_info=data_info,
        do_resample=True,
        res_param=res_param,
        do_medial_rm=True,
        medial_param=medial_param,
        normalize="zscore",
        dtype=np.float32,
        njobs=3,
        verbose=5,
        json_path=json_path,
        overwrite=False,
        log_num='cifti_demo',
)

### CIFTI Subcortical Volume Data

CIFTI subcortical data are extracted from the CIFTI BrainModels axis and prepared as volume-derived `time x voxel` matrices. Use `prep.prep_volume` with a CIFTI input validation dictionary.

| Parameter | Meaning |
|---|---|
| `work_dir` | Root output directory. |
| `input_info` | Validation dictionary returned by `prep.input_validation_nii`. |
| `cifti_region` | Subcortical region selector. Common abbreviations are `L`, `R`, and `BRAIN_STEM`. You can also use a full CIFTI structure name such as `CIFTI_STRUCTURE_AMYGDALA_LEFT`, a brief keyword such as `AMYGDALA`, or a list of structures. |
| `roi_name` | ROI label used in output folders and filenames. For GUI-compatible output, use labels such as `subcortical-L`, `subcortical-R`, and `subcortical-BRAIN_STEM`. |
| `do_resample` | Whether to resample the extracted volume to another grid. For CIFTI subcortical preparation, `False` is commonly recommended unless a specific volume grid is required. |
| `res_param` | Volume resampling dictionary used only when `do_resample=True`. For CIFTI examples without resampling, set it to `None`. |
| `mask` | External mask for NIfTI input. For CIFTI subcortical extraction, use `None` so the header-derived subcortical mask is used. |
| `save_resample_mask` | If `True`, saves a representative 3D mask in `work_dir/masks`. |
| `normalize` | Normalization method. Use `zscore`, `demean`, or `None` for raw values. |
| `dtype` | Numeric type for saved arrays. |
| `njobs` | Number of parallel workers used for file-wise preparation. |
| `verbose` | Joblib verbosity level. |
| `json_path` | JSON log path used to record processing settings. |
| `overwrite` | Whether to overwrite matching JSON records. |
| `log_num` | Identifier appended to the progress log filename. |

The output folders for the three standard subcortical groups are `volume-subcortical-L`, `volume-subcortical-R`, and `volume-subcortical-BRAIN_STEM`.

In [ ]:
for structure_name in ["L", "R", "BRAIN_STEM"]:
    prep.prep_volume(
        work_dir,
        input_info=data_info,
        cifti_region=structure_name,
        roi_name=f"subcortical-{structure_name}",
        do_resample=False,
        res_param=None,
        mask=None,
        save_resample_mask=True,
        normalize="zscore",
        dtype=np.float32,
        njobs=2,
        verbose=5,
        json_path=json_path,
        overwrite=False,
        log_num="cifti_subcortical",
    )

### NIfTI Volume Data

NIfTI preparation uses `prep.input_validation_nii` followed by `prep.prep_volume`. Unlike CIFTI subcortical extraction, NIfTI features are selected with a 3D mask or, if no mask is provided, by retaining finite non-zero voxels.

| Parameter | Meaning |
|---|---|
| `work_dir` | Root output directory. |
| `input_info` | Validation dictionary returned by `prep.input_validation_nii`. |
| `cifti_region` | Not used for NIfTI input. Set to `None`. |
| `roi_name` | Name of the extracted volume ROI, written into the output folder as `volume-<roi_name>`. |
| `do_resample` | If `True`, resamples the input NIfTI before masking. |
| `res_param` | Volume resampling dictionary. |
| `res_param['ref']` | Reference NIfTI image used when `mode='ref'`. The prepared image is resliced to match this reference grid. |
| `res_param['interpolation']` | Resampling interpolation method. Use `nearest` for masks or labels; use continuous interpolation only when appropriate for image intensity data. |
| `res_param['vol_size']` | Target voxel size used when `mode='vox'`; also passed through the resampling helper in reference-based workflows. |
| `res_param['mode']` | `ref` means match the grid of `ref`; `vox` means resample based on target voxel size. |
| `res_param['res']` | Short resolution label written into filenames as `res-<label>`. |
| `mask` | Path to a 3D NIfTI mask. Voxels with non-zero mask values are kept. If `None`, ``fMRI-HA`` builds a finite non-zero mask from the data. |
| `save_resample_mask` | If `True`, saves a representative 3D mask in `work_dir/masks`. |
| `normalize` | Normalization method. Use `zscore`, `demean`, or `None` for raw values. |
| `dtype` | Numeric type for saved arrays. |
| `njobs` | Number of parallel workers used for file-wise preparation. |
| `verbose` | Joblib verbosity level. |
| `json_path` | JSON log path used to record processing settings. |
| `overwrite` | Whether to overwrite matching JSON records. |
| `log_num` | Identifier appended to the progress log filename. |

In [ ]:
nifti_dir = Path("/path/to/raw_data/nifti")
work_dir = Path("/path/to/workdir_nifti")
json_path = work_dir / "logs" / "prep_log.json"

sub_list = [
    "sub-rid000005",
    "sub-rid000014",
    "sub-rid000029",
]

nifti_list = [
    nifti_dir / f"{sub}_task-raiders_acq-8ch336vol_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
    for sub in sub_list
]

nifti_info = prep.input_validation_nii(
    file_list=nifti_list,
    ses="raiders",
    bids_parse=True,
    subject_key=None,
    check_dim=True,
    group_by="run",
    role="train",
    njobs=2,
    verbose=True,
)

prep.write_json_input_validate(nifti_info, json_path, overwrite=False)

res_param = {
    "ref": "/path/to/HCPex_2mm_cortical_mask.nii",
    "interpolation": "nearest",
    "vol_size": 3,
    "mode": "ref",
    "res": "3mm",
}

mask = "/path/to/HCPex_2mm_cortical_mask.nii"

prep.prep_volume(
    work_dir,
    input_info=nifti_info,
    cifti_region=None,
    roi_name="cortical",
    do_resample=True,
    res_param=res_param,
    mask=mask,
    save_resample_mask=True,
    normalize="zscore",
    dtype=np.float32,
    njobs=2,
    verbose=5,
    json_path=json_path,
    overwrite=False,
    log_num="nifti_demo",
)

### GIFTI Surface Data

GIFTI preparation is surface-only. Each subject is represented by a dictionary whose keys identify the hemispheres. Use `{'l': left_gifti, 'r': right_gifti}` when both hemispheres are present.

The preparation sequence is similar to CIFTI surface preparation:

1. `prep.input_validation_gii` validates hemisphere files and checks dimensions.
2. `prep.prep_surf` resamples, masks, normalizes, and saves hemisphere-specific `.npy` files.

Important GIFTI-specific points:

| Item | Meaning |
|---|---|
| `file_list` | A list of hemisphere dictionaries, not a simple list of paths. |
| `source` in `res_param` | The GIFTI source surface space, such as `fslr-ico128`, `fsaverage`, or another supported space. |
| `data` in `res_param` | Mapper for the GIFTI source space to the target analysis space. |
| `Structure` in downstream workflow | Prepared GIFTI outputs are normally used as `hemi-L` and `hemi-R`. |

Other `prep.prep_surf` parameters have the same meanings described in the CIFTI surface section.

In [ ]:
gifti_dir = Path("/path/to/raw_data/gifti")
work_dir = Path("/path/to/workdir_gifti")
json_path = work_dir / "logs" / "prep_log.json"

sub_list = [
    "sub-rid000005",
    "sub-rid000014",
    "sub-rid000029",
]

train_list = []
for sub in sub_list:
    train_list.append(
        {
            "l": gifti_dir / f"{sub}_task-raiders_acq-8ch336vol_run-01_hemi-L_space-fsaverage_bold.func.gii",
            "r": gifti_dir / f"{sub}_task-raiders_acq-8ch336vol_run-01_hemi-R_space-fsaverage_bold.func.gii",
        }
    )

gifti_info = prep.input_validation_gii(
    file_list=train_list,
    ses="raiders",
    bids_parse=True,
    subject_key=None,
    check_dim=True,
    group_by="run",
    role="train",
    njobs=2,
    verbose=True,
)

prep.write_json_input_validate(gifti_info, json_path, overwrite=False)

res_param = {
    "source": "fslr-ico57",
    "source_mask": None,
    "target": "onavg-ico32",
    "target_mask": None,
    "data": None,
    "desc": "fslr-ico57 whole to onavg-ico32 whole",
    "res": "onavg32",
}

medial_param = {
    "space": "onavg-ico32",
    "data": None,
    "roi_name": "cortical",
    "desc": "onavg-ico32 cortical",
}

prep.prep_surf(
    work_dir,
    input_info=gifti_info,
    do_resample=True,
    res_param=res_param,
    do_medial_rm=True,
    medial_param=medial_param,
    normalize="zscore",
    dtype=np.float32,
    njobs=3,
    verbose=5,
    json_path=json_path,
    overwrite=False,
    log_num="gifti_demo",
)

### Concatenating Prepared Files

After preparing individual runs, use `prep.combine_file` to concatenate multiple prepared `.npy` files along the time dimension. Concatenation is performed separately for each subject and each requested structure.

| Parameter | Meaning |
|---|---|
| `work_dir` | Root directory containing subject folders. |
| `sub_list` | Subject IDs to process. These must match the subject folder names. |
| `step` | Processing step folder to search, usually `resample` for prepared response data. |
| `modality` | Modality folder under `step`, commonly `response` or `connectivity`. |
| `structures` | Structure folders to concatenate, such as `hemi-L`, `hemi-R`, `volume-subcortical-L`, or `volume-cortical`. |
| `ses_name` | Session label assigned to the concatenated output file. |
| `njobs` | Number of parallel workers across subject-structure jobs. |
| `method` | Optional normalization applied after concatenation. Use `None` if each run was already normalized and you do not want to normalize again. |
| `json_path` | JSON log path used to record the concatenation step. |
| `dtype` | Output numeric type used if post-concatenation normalization is applied. |
| `overwrite` | Whether to overwrite matching JSON records. |
| `log_num` | Identifier appended to the progress log filename. |
| `**file_filter` | Key-value filters used to select files. For example, `{'run': ['01', '02']}` combines only run 01 and run 02 files. |

`file_filter` is matched against BIDS-like filename entities. Add more filters, such as `space`, `roi`, `res`, or `set`, when the folder contains multiple candidate files.

In [ ]:
work_dir = Path("/path/to/workdir_cifti")
json_path = work_dir / "logs" / "prep_log.json"

sub_list = [
    "sub-rid000005",
    "sub-rid000014",
    "sub-rid000029",
]

file_filter = {
    "run": ["01", "02"],
}

prep.combine_file(
    work_dir,
    sub_list,
    step="resample",
    modality="response",
    structures=[
        "hemi-L",
        "hemi-R",
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    ses_name="comb12",
    njobs=2,
    method=None,
    json_path=json_path,
    dtype=np.float32,
    overwrite=False,
    log_num="combine_runs",
    **file_filter,
)

## GUI Workflow

Start the GUI with the code below. The GUI is designed for whole-brain searchlight hyperalignment. In the current version, the data preparation panel supports CIFTI and GIFTI directly; NIfTI volume preparation is available through the scripts workflow above.

In [ ]:
from fmriha.gui import gui
gui()

```{image} pic/data_prep/fig_2.png
:alt: GUI launch windows
:width: 700px
```

Before opening data preparation, set the global GUI fields:

| GUI Field | Meaning |
|---|---|
| `Work Dir` | Root directory where configuration files, logs, prepared arrays, masks, and downstream outputs are written. |
| `n_jobs Parameters -> Data Preparation` | Number of parallel workers used for validation and data preparation. Use smaller values when memory is limited. |
| `Space and Searchlight Configuration -> Surface -> Space` | Target surface space for cortical data preparation, such as `onavg-ico32`. |
| `Space and Searchlight Configuration -> Surface -> Surface Searchlight Radius` | Searchlight radius used later by hyperalignment stages; it is saved with the workflow configuration. |
| `Space and Searchlight Configuration -> Volume -> Volume Searchlight Radius` | Volume searchlight radius used later for subcortical searchlights. |
| `Space and Searchlight Configuration -> Volume -> Threshold` | Threshold used later during volume searchlight generation. |
| `Space and Searchlight Configuration -> Volume -> CIFTI Reference` | Reference CIFTI file used later to derive subcortical masks for volume searchlights. Provide this when subcortical structures are part of the workflow. |

Insert a screenshot for global setup here. The image should highlight **Work Dir**, the **n_jobs Parameters** dialog, and the **Space and Searchlight Configuration** window.

```{image} pic/data_prep/fig_gui_global_settings.png
:alt: GUI global settings for work directory, jobs, and searchlight configuration
:width: 750px
```

Open **Data Preparation** and choose one of the three tabs: **Template Data**, **Transformation Data**, or **Alignment Data**. Each tab can contain one or more modules. A module corresponds to one input dataset and one preparation configuration.

| GUI Field | Meaning |
|---|---|
| `Data Path` | Table of input files. Add all files belonging to the selected module. |
| `Data Type` | Input format. Choose `CIFTI` for CIFTI files or `GIFTI` for GIFTI files. |
| `Structure` | Output structures to prepare. Surface choices include `hemi-L` and `hemi-R`; CIFTI volume choices include `subcortical-L`, `subcortical-R`, and `subcortical-BRAIN_STEM`. |
| `Source Space` | Surface space of the input data. Used for surface resampling. |
| `Target Space` | Displayed from the global surface-space setting. Prepared cortical data are mapped to this space. |
| `Source Medial Wall Removed` | Enable this when source surface data already have the medial wall removed. The selected file is used as the source mask for mapping. |
| `Normalize` | `raw` keeps values unchanged, `demean` removes each feature mean over time, and `zscore` standardizes each feature over time. |
| `Sessions` | Session label written into output filenames. This corresponds to `ses` in the Python API. |
| `Group By` | Filename entity used for grouping and dimension checks, commonly `run`. |
| `Role` | Dataset role written into output filenames, such as `train`, `test`, or `align`. |
| `BIDS File` | Enables BIDS-like filename parsing. When enabled, subject IDs and entities are parsed from filenames. |
| `Subject Key` | Used when `BIDS File` is disabled. It tells the parser where to find the subject ID in non-BIDS filenames. |
| `Float Type` | Numeric precision for outputs. `float32` is recommended for most workflows; `float64` uses more memory. |
| `+` | Adds another module to the current data-preparation tab. |
| `-` | Removes a module. |
| `Same as Tmpl. Data` | In the Transformation or Alignment tab, reuses the Template Data settings and subject list. |
| `Same as T Data` | In the Alignment tab, reuses the Transformation Data settings and subject list. |

Insert a screenshot of the data preparation window here. The image should show the **Template Data**, **Transformation Data**, and **Alignment Data** tabs, a module panel, the **Data Path** table, and the common module parameters.

```{image} pic/data_prep/fig_gui_data_prep_window.png
:alt: GUI data preparation window and module fields
:width: 800px
```

**Surface preparation in the GUI: CIFTI cortex and GIFTI**

In the GUI, CIFTI cortical surface data and GIFTI surface data can be explained together because they use the same surface-oriented fields. The main difference is the selected **Data Type** and the files added to **Data Path**.

For CIFTI surface data, set **Data Type** to `CIFTI` and add `.dtseries.nii` or `.dscalar.nii` files. For GIFTI surface data, set **Data Type** to `GIFTI` and add the corresponding left/right `.func.gii` files. Then select `hemi-L`, `hemi-R`, or both in **Structure**.

Recommended surface setup:

1. Add the CIFTI or GIFTI files to **Data Path**.
2. Set **Data Type** to `CIFTI` or `GIFTI`.
3. Select `hemi-L`, `hemi-R`, or both in **Structure**.
4. Set **Source Space** to the space of the input surface data.
5. Confirm **Target Space** from **Space and Searchlight Configuration -> Surface -> Space**.
6. Enable **Source Medial Wall Removed** only if the source data already exclude medial-wall vertices, then provide the matching source mask file.
7. Fill **Normalize**, **Sessions**, **Group By**, **Role**, **BIDS File**, **Subject Key**, and **Float Type**.
8. Save the configuration or click **RUN!** after selecting the workflow stages to execute.

Internally, CIFTI surface data are validated with `prep.input_validation_nii`, while GIFTI surface data are validated with `prep.input_validation_gii`. Both are then prepared with `prep.prep_surf`.

Insert paired GUI screenshots for surface preparation here. The first image should show **Data Type = CIFTI** with selected `hemi-L`/`hemi-R`; the second should show **Data Type = GIFTI** with left/right GIFTI files. The two screenshots should make clear that the surface parameters are otherwise the same.

```{image} pic/data_prep/fig_gui_surface_cifti_gifti.png
:alt: GUI surface setup for CIFTI cortical and GIFTI data
:width: 850px
```

**Volume preparation in the GUI: CIFTI subcortex**

Use the GUI volume workflow for the subcortical component of CIFTI files. Add the CIFTI files, set **Data Type** to `CIFTI`, and select one or more of `subcortical-L`, `subcortical-R`, and `subcortical-BRAIN_STEM` in **Structure**. These choices correspond to `cifti_region='L'`, `cifti_region='R'`, and `cifti_region='BRAIN_STEM'` in the scripts workflow.

The CIFTI volume workflow uses the CIFTI header-derived subcortical mask and does not require an external mask. If later stages will generate subcortical searchlights, also set **Space and Searchlight Configuration -> Volume -> CIFTI Reference** before running.

NIfTI volume preparation is not exposed in the current GUI data preparation panel. Use the NIfTI scripts workflow in this notebook for NIfTI data.

Insert a CIFTI volume GUI screenshot here. The image should show **Data Type = CIFTI**, selected subcortical structures, and the global **CIFTI Reference** field used for later volume searchlight generation.

```{image} pic/data_prep/fig_gui_volume_cifti.png
:alt: GUI volume setup for CIFTI subcortical data
:width: 800px
```

**Combining files in the GUI**

Use **Combine Files** when each subject has multiple prepared runs or modules that should be concatenated before later analysis. The subject lists across modules must be consistent.

| GUI Field | Meaning |
|---|---|
| `New Session Name` | Session label assigned to the concatenated output file. This corresponds to `ses_name`. |
| `Step` | Folder to search inside each subject directory, usually `resample`. |
| `Modality` | Modality folder under the step, commonly `response`. |
| `Structure` | One or more structure folders to concatenate, such as `hemi-L`, `hemi-R`, or `volume-subcortical-L`. |
| `File Filter` | Key-value filters for selecting files to concatenate, such as `run=01` and `run=02`. Add more rows when you need to restrict `space`, `roi`, `res`, `set`, or other filename entities. |

Internally, the GUI calls `prep.combine_file` for the selected structures and filters.

Insert a GUI screenshot for file concatenation here. The image should show the **Combine Files** panel with **New Session Name**, **Step**, **Modality**, selected structures, and multiple file-filter rows such as `run=01` and `run=02`.

```{image} pic/data_prep/fig_gui_combine_files.png
:alt: GUI combine files panel
:width: 800px
```

Insert a final GUI execution screenshot here. The image should show saving the configuration, selecting stages to run, clicking **RUN!**, and optionally opening the monitor window to inspect progress logs.

```{image} pic/data_prep/fig_gui_run_monitor.png
:alt: GUI run and monitor workflow
:width: 800px
```

## Practical Checks Before Moving On

Before running response hyperalignment, check the following:

- The subject list is the same across all modules that will be combined or analyzed together.
- Prepared files exist under the expected structure folders.
- Output filenames contain the expected `ses`, `run`, `space`, `roi`, `res`, `desc`, and `set` entities.
- Surface data from different subjects have the same number of vertices after resampling and masking.
- Volume data from different subjects have the same number of voxels after masking.
- Normalization has been applied exactly once unless you intentionally normalize again after concatenation.
- The JSON log at `logs/prep_log.json` records the validation and preparation settings you intended to use.
- For GUI workflows, the saved `.fmriha` configuration in `Work Dir` matches the visible settings before clicking **RUN!**.